In [0]:
bookings_df = spark.read.csv(
    "/Volumes/learn_databricks_7405613655466590/default/datasets/bookings.csv",
    header=True,
    inferSchema=True
)

flights_df = spark.read.csv(
    "/Volumes/learn_databricks_7405613655466590/default/datasets/Flights.csv",
    header=True,
    inferSchema=True
)

passengers_df = spark.read.option(
    'multiline',
    'true'
    ).json(
    "/Volumes/learn_databricks_7405613655466590/default/datasets/passenger_preferences.json"
)

In [0]:
bookings_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_bookings")

flights_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_flights")

passengers_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_passengers")

In [0]:
silver_bookings = bookings_df.dropDuplicates()

silver_flights = flights_df.dropDuplicates()

silver_passengers = passengers_df.dropDuplicates()

In [0]:
silver_bookings.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_bookings")

silver_flights.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_flights")

silver_passengers.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_passengers")

In [0]:
from pyspark.sql.functions import sum

gold_airline_revenue = silver_bookings.join(
    silver_flights,
    "flight_id"
).groupBy(
    "airline"
).agg(
    sum("ticket_price").alias("revenue")
)

gold_airline_revenue.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_airline_revenue")

In [0]:
from pyspark.sql.functions import concat_ws

gold_route_revenue = silver_bookings.join(
    silver_flights,
    "flight_id"
).groupBy(
    "from_city",
    "to_city"
).agg(
    sum("ticket_price").alias("revenue")
)

gold_route_revenue.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_route_revenue")

In [0]:
gold_passenger_preference = silver_passengers.select(
    "passenger_name",
    "meal",
    "seat"
)

gold_passenger_preference.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_passenger_preference")

In [0]:
gold_flight_delay = silver_flights.select(
    "flight_id",
    "status"
)

gold_flight_delay.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_flight_delay")

In [0]:
from pyspark.sql.functions import rank
from pyspark.sql.window import Window

flight_revenue = silver_bookings.groupBy(
    "flight_id"
).agg(
    sum("ticket_price").alias("revenue")
)

window_spec = Window.orderBy(
    flight_revenue["revenue"].desc()
)

gold_top_revenue_flights = flight_revenue.withColumn(
    "rank",
    rank().over(window_spec)
)

gold_top_revenue_flights.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_top_revenue_flights")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
display(spark.table("gold_airline_revenue"))
display(spark.table("gold_route_revenue"))
display(spark.table("gold_passenger_preference"))
display(spark.table("gold_flight_delay"))
display(spark.table("gold_top_revenue_flights"))

airline,revenue
Air India,68000
Akasa,62000
Indigo,90000
Vistara,71500


from_city,to_city,revenue
Delhi,Chennai,28000
Goa,Delhi,7000
Hyderabad,Delhi,39000
Delhi,Mumbai,7500
Mumbai,Chennai,9000
Pune,Delhi,10000
Bangalore,Mumbai,23500
Delhi,Hyderabad,18000
Kolkata,Bangalore,10500
Chennai,Bangalore,25000


passenger_name,meal,seat
Priya Reddy,Non-Veg,Aisle
Neha Singh,Veg,Window
Nisha Reddy,Non-Veg,Window
Kiran Rao,Veg,Aisle
Rohit Sharma,Veg,Aisle
Ayesha Khan,Jain,Window
Divya Iyer,Jain,Window
Arjun Verma,Veg,Middle
Meera Nair,Jain,Window
Rahul Sharma,Veg,Window


flight_id,status
F104,Cancelled
F108,On Time
F112,Cancelled
F114,Delayed
F107,On Time
F109,Delayed
F111,On Time
F102,Delayed
F101,On Time
F103,On Time


flight_id,revenue,rank
F101,39000,1
F103,38000,2
F109,28000,3
F107,26500,4
F105,25000,5
F113,24000,6
F110,23500,7
F115,18000,8
F111,16000,9
F114,10500,10


In [0]:
bookings_df.write.format("delta").mode("overwrite") \
    .save("/Volumes/learn_databricks_7405613655466590/generated_data/bronze/bookings")

flights_df.write.format("delta").mode("overwrite") \
    .save("/Volumes/learn_databricks_7405613655466590/generated_data/bronze/flights")

passengers_df.write.format("delta").mode("overwrite") \
    .save("/Volumes/learn_databricks_7405613655466590/generated_data/bronze/passengers")

In [0]:
silver_bookings = bookings_df.dropDuplicates()

silver_flights = flights_df.dropDuplicates()

silver_passengers = passengers_df.dropDuplicates()

In [0]:
silver_bookings.write.format("delta").mode("overwrite") \
    .save("/Volumes/learn_databricks_7405613655466590/generated_data/silver/bookings")

silver_flights.write.format("delta").mode("overwrite") \
    .save("/Volumes/learn_databricks_7405613655466590/generated_data/silver/flights")

silver_passengers.write.format("delta").mode("overwrite") \
    .save("/Volumes/learn_databricks_7405613655466590/generated_data/silver/passengers")

In [0]:
gold_airline_revenue.write.format("delta").mode("overwrite") \
    .save("/Volumes/learn_databricks_7405613655466590/generated_data/gold/airline_revenue_report")

gold_route_revenue.write.format("delta").mode("overwrite") \
    .save("/Volumes/learn_databricks_7405613655466590/generated_data/gold/route_revenue_report")

gold_passenger_preference.write.format("delta").mode("overwrite") \
    .save("/Volumes/learn_databricks_7405613655466590/generated_data/gold/passenger_preference_report")

gold_flight_delay.write.format("delta").mode("overwrite") \
    .save("/Volumes/learn_databricks_7405613655466590/generated_data/gold/flight_delay_report")

gold_top_revenue_flights.write.format("delta").mode("overwrite") \
    .save("/Volumes/learn_databricks_7405613655466590/generated_data/gold/top_revenue_flights_report")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
